# 🎯 8주차 킬러 과제(난이도 조정판) — AI Agent Workflow 실무 구축

> **강의**: 8주차 AI Agent Workflow & 자동화 파이프라인 설계  
> **제출 형식**: 본 `.ipynb` + n8n export + OpenClaw 로그
---

## 📌 과제 개요


### 수업 매핑
| 문항 | 배점 | 관련 수업 |
|---|---:|---|
| Q1 | 30 | Part 2 — n8n Docker/Webhook/HTTP/IF/Set/Schedule |
| Q2 | 35 | Part 3 — OpenClaw 설치 + Google OAuth + Gmail API |

### 📊 채점 PASS 기준
| 문항 | 제출물 | PASS |
|---|---|---|
| Q1 | `workflow_q1.json` | 필수 노드 4종 + 연결 + 표현식 |
| Q2 | `openclaw_setup.json` + `gmail_log.jsonl` + `gmail_summary.md` | 본인 이메일 매칭 + 8건 로그 + 기본 마스킹 |


---

# 🔧 Q1. n8n 워크플로우 빌드 (30점)

**목표:** n8n Docker 기동 → 간단한 기상 조회/경보 워크플로우 JSON export.

권장 체인:  
[Schedule] → [HTTP: Weather API] → [Set] → [IF] → [NoOp]

1. 워크플로우 이름: `[{본인이름}]_Weather_Alert`
2. 노드 4종 포함: scheduleTrigger, httpRequest, set, if (NoOp는 권장)
3. HTTP URL은 **실제 외부 HTTPS** (localhost/example.com 금지)
4. n8n 표현식 `={{ $json... }}` 1회 이상
5. Schedule → HTTP → Set → IF 연결
6. Schedule 주기: **10분 이상**(기존 5분 → 완화)
7. Set 노드에서 값 매핑 1개 이상(기존 2개 → 완화)

**제출:** `./submit/workflow_q1.json`

> 팁: Open-Meteo 무료 API 사용 가능
- URL 예: `https://api.open-meteo.com/v1/forecast`
- Query: latitude/longitude/current 등


### 자동채점

In [ ]:
# Q1 자동 채점 (완화판)
Q1_PATH = SUBMIT_DIR / "workflow_q1.json"
q1_result = {"pass": False, "score": 0, "checks": {}, "errors": []}

def _check(r, k, cond, w, m=""):
    r["checks"][k] = {"pass": bool(cond), "weight": w, "msg": m}
    if cond: r["score"] += w
    else: r["errors"].append(f"{k}: {m}")

if Q1_PATH.exists():
    wf = json.loads(Q1_PATH.read_text(encoding="utf-8"))
    name, nodes, conns = wf.get("name",""), wf.get("nodes",[]), wf.get("connections",{})
    _check(q1_result, "name", name == f"[{STUDENT_NAME}]_Weather_Alert", 6, f"이름 불일치: {name}")

    types = [n.get("type","") for n in nodes]
    for k, t in [("scheduleTrigger","n8n-nodes-base.scheduleTrigger"),
                 ("httpRequest","n8n-nodes-base.httpRequest"),
                 ("set","n8n-nodes-base.set"),
                 ("if","n8n-nodes-base.if")]:
        _check(q1_result, f"node_{k}", t in types, 5, f"{t} 누락")

    http_ok = any(
        (n.get("parameters",{}).get("url","") or "").startswith("https://") and
        "localhost" not in (n.get("parameters",{}).get("url","") or "") and
        "example.com" not in (n.get("parameters",{}).get("url","") or "")
        for n in nodes if n.get("type") == "n8n-nodes-base.httpRequest")
    _check(q1_result, "http_real_url", http_ok, 4, "외부 HTTPS URL 필요")

    flat = json.dumps(wf, ensure_ascii=False)
    _check(q1_result, "expression", bool(re.search(r"=\{\{\s*\$json", flat)) or bool(re.search(r"\{\{\s*\$json", flat)),
           4, "표현식 필요 (={{ $json... }})")

    def _nb(tp): return [n["name"] for n in nodes if n.get("type")==tp]
    def _next(s):
        return [c.get("node") for b in conns.get(s,{}).get("main",[]) for c in (b or [])]
    chain = False
    for sch in _nb("n8n-nodes-base.scheduleTrigger"):
        for h in _next(sch):
            if any(n.get("name")==h and n.get("type")=="n8n-nodes-base.httpRequest" for n in nodes):
                for s in _next(h):
                    if any(n.get("name")==s and n.get("type")=="n8n-nodes-base.set" for n in nodes):
                        for i in _next(s):
                            if any(n.get("name")==i and n.get("type")=="n8n-nodes-base.if" for n in nodes):
                                chain = True
    _check(q1_result, "chain", chain, 5, "Schedule→HTTP→Set→IF 체인")

    sch_ok = False
    for n in nodes:
        if n.get("type") == "n8n-nodes-base.scheduleTrigger":
            for r in n.get("parameters",{}).get("rule",{}).get("interval",[]):
                if r.get("field")=="minutes" and int(r.get("minutesInterval",0) or 0) >= 10: sch_ok = True
                if r.get("field") in ("hours","days","weeks","months"): sch_ok = True
    _check(q1_result, "schedule_interval", sch_ok, 2, "10분 이상")

    set_ok = False
    for n in nodes:
        if n.get("type") == "n8n-nodes-base.set":
            v = n.get("parameters",{}).get("values",{})
            cnt = sum(len(x) for x in v.values() if isinstance(x,list))
            cnt += len(n.get("parameters",{}).get("assignments",{}).get("assignments",[]))
            if cnt >= 1: set_ok = True
    _check(q1_result, "set_values", set_ok, 4, "Set 값 1개 이상")
else:
    q1_result["errors"].append(f"파일 없음: {Q1_PATH}")

q1_result["pass"] = q1_result["score"] >= 24
print(json.dumps(q1_result, ensure_ascii=False, indent=2))


---

# 🛠 Q2. OpenClaw + Google OAuth + Gmail (35점)

**목표:** OpenClaw 설치 → Google OAuth → Gmail 최근 7일 메일 **8건** 조회 → 로그 + 요약 제출.

## 제출물
- `./submit/gmail_log.jsonl` — **메일 8건**
- `./submit/gmail_summary.md` — **200자 이상** 요약

## 🔒 이메일 규칙
- 이메일: `jo***@domain.com` 형태
- snippet: 앞 80자만

### 실행 가이드
```bash
npm i -g openclaw
openclaw --version
openclaw auth --scopes gmail.readonly
openclaw gmail list --since 7d --max 8 --format json > /tmp/raw.json
```



### 자동채점

In [ ]:
# Q2 자동 채점 (완화판)
Q2_SETUP = SUBMIT_DIR / "openclaw_setup.json"
Q2_LOG = SUBMIT_DIR / "gmail_log.jsonl"
Q2_SUMMARY = SUBMIT_DIR / "gmail_summary.md"

q2_result = {"pass": False, "score": 0, "checks": {}, "errors": []}
def _q2(k, c, w, m=""):
    q2_result["checks"][k] = {"pass": bool(c), "weight": w, "msg": m}
    if c: q2_result["score"] += w
    else: q2_result["errors"].append(f"{k}: {m}")

setup = None
if Q2_SETUP.exists():
    try: setup = json.loads(Q2_SETUP.read_text(encoding="utf-8"))
    except Exception as e: q2_result["errors"].append(f"setup 파싱 실패: {e}")
else: q2_result["errors"].append(f"없음: {Q2_SETUP}")

if setup:
    _q2("openclaw_version", bool(re.match(r"^\d+\.\d+\.\d+", str(setup.get("openclaw_version","")))), 3, "semver")
    _q2("node_version", str(setup.get("node_version","")).startswith("v"), 2, "v... 형식")
    _q2("gmail_scope", any("gmail" in s for s in setup.get("scopes",[])), 3, "gmail scope")
    _q2("email_match", setup.get("authorized_email","").lower() == STUDENT_EMAIL.lower(), 7,
        f"authorized_email != {STUDENT_EMAIL}")
    _q2("auth_time", "authorized_at" in setup and len(str(setup["authorized_at"])) >= 10, 2, "timestamp 필요")

logs = []
if Q2_LOG.exists():
    for l in Q2_LOG.read_text(encoding="utf-8").splitlines():
        if l.strip():
            try: logs.append(json.loads(l))
            except Exception: pass
    _q2("log_count", len(logs) == 8, 6, f"8건 필요 (현재 {len(logs)})")
    req = {"message_id","thread_id","from_masked","to_masked","subject","snippet_masked","received_at"}
    _q2("log_schema", all(req.issubset(set(r.keys())) for r in logs), 3, "스키마 누락")
    if logs:
        # 평문 이메일 누출 검사
        flat = json.dumps(logs, ensure_ascii=False)
        leaks = [e for e in re.findall(r"[a-zA-Z0-9._%+-]{3,}@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", flat) if "***" not in e]
        _q2("masking_applied", len(leaks) == 0, 4, f"평문 이메일 {len(leaks)}건")
else: q2_result["errors"].append(f"없음: {Q2_LOG}")

if Q2_SUMMARY.exists():
    _q2("summary_length", len(Q2_SUMMARY.read_text(encoding="utf-8")) >= 200, 5, "200자 이상")
else: q2_result["errors"].append(f"없음: {Q2_SUMMARY}")

q2_result["pass"] = q2_result["score"] >= 28
print(json.dumps(q2_result, ensure_ascii=False, indent=2))


---

## 📦 제출 폴더 구조

```
./submit/
├── workflow_q1.json            ← Q1
├── openclaw_setup.json         ← Q2
├── gmail_log.jsonl             ← Q2 (마스킹된 8건)
├── gmail_summary.md            ← Q2 (200자+)
```
